# Pikachu robust RL — Colab T4
This notebook pins both repositories, verifies the production engine, resumes only verified episode-boundary checkpoints from Drive, and keeps validation separate from any sealed final set.

Private repository setup: create a fine-grained GitHub token with read-only Contents access to `jimin326/skku_pikachu`, then add it to Colab Secrets as `GITHUB_TOKEN` and enable notebook access. The token is passed through a temporary Git HTTP header and is never written to the clone URL or Git config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_URL = 'https://github.com/jimin326/skku_pikachu.git'
PROJECT_ROOT = '/content/skku_pikachu'
PROJECT_REF = 'robust-rl-colab'  # pinned branch containing the RL system
GAME_URL = 'https://github.com/SKKU-x-HYU-SW-Competition/leonyi-volleyball.git'
GAME_COMMIT = '1f3cecb90aca174ffc42ac6be4c384cc725d9e91'
RECOVERY = '/content/drive/MyDrive/pikachu_rl/checkpoints'


In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import userdata
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a read-only GITHUB_TOKEN in Colab Secrets and enable notebook access') from exc
if not github_token:
    raise RuntimeError('Colab Secret GITHUB_TOKEN is empty')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env = os.environ.copy()
git_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'Authorization: Basic {basic_auth}')
project_path = Path(PROJECT_ROOT)
if not (project_path / '.git').is_dir():
    if project_path.exists() and any(project_path.iterdir()):
        raise RuntimeError(f'{project_path} exists but is not a Git checkout; restart the runtime or clear that directory')
    subprocess.run(['git','clone','--branch',PROJECT_REF,'--single-branch',PROJECT_URL,PROJECT_ROOT], check=True, env=git_env)
subprocess.run(['git','-C',PROJECT_ROOT,'fetch','origin',PROJECT_REF], check=True, env=git_env)
subprocess.run(['git','-C',PROJECT_ROOT,'checkout','-B',PROJECT_REF,'origin/'+PROJECT_REF], check=True, env=git_env)
github_token = basic_auth = git_env = None  # discard notebook references to credentials
game_root = Path('/content/leonyi-volleyball')
if not (game_root / '.git').is_dir():
    if game_root.exists() and any(game_root.iterdir()):
        raise RuntimeError(f'{game_root} exists but is not a Git checkout; restart the runtime or clear that directory')
    subprocess.run(['git','clone',GAME_URL,str(game_root)], check=True)
subprocess.run(['git','-C',str(game_root),'checkout',GAME_COMMIT], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-rl.txt'], check=True)
subprocess.run(['node','scripts/setup_rl_engine.mjs',str(game_root)], check=True)
print('Prepared', PROJECT_REF, 'at', subprocess.run(['git','rev-parse','HEAD'], cwd=PROJECT_ROOT, check=True, text=True, capture_output=True).stdout.strip())


In [ ]:
!node bot-dev/rl/physics_clamp_smoke.mjs
!node bot-dev/rl/production_differential.mjs --game-root /content/leonyi-volleyball
!node bot-dev/rl/env_smoke.mjs
!python bot-dev/rl/bridge_smoke.py
!python bot-dev/rl/ppo_tests.py
!python bot-dev/rl/eval/test_schema.py
!python bot-dev/rl/eval/test_stats.py
!node bot-dev/rl/eval/paired_eval_smoke.mjs
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime before training'
print(torch.cuda.get_device_name(0))


In [ ]:
# Optional v4 behavior-cloning initialization. Increase decisions for a real run.
RUN_BC = False
if RUN_BC:
    !node bot-dev/rl/collect_bc.mjs --decisions=500000 --output=/content/drive/MyDrive/pikachu_rl/bc/v4.jsonl
    !python bot-dev/rl/bc_pretrain.py /content/drive/MyDrive/pikachu_rl/bc/v4.jsonl /content/drive/MyDrive/pikachu_rl/bc/v4_ff.pt --epochs=10 --device=cuda


In [ ]:
import hashlib, json, os
latest = os.path.join(RECOVERY, 'latest.json')
resume_args = []
if os.path.exists(latest):
    pointer = json.load(open(latest))
    checkpoint = os.path.join(RECOVERY, pointer['checkpoint'])
    assert hashlib.sha256(open(checkpoint, 'rb').read()).hexdigest() == pointer['sha256']
    resume_args = ['--resume', checkpoint]
elif RUN_BC:
    resume_args = ['--initial-model', '/content/drive/MyDrive/pikachu_rl/bc/v4_ff.pt']
resume_args


In [ ]:
# Benchmark 8/16/32 total envs first; Node physics/IPC, not VRAM, is usually the bottleneck.
from pathlib import Path
import subprocess, sys
project_root = Path(PROJECT_ROOT)
train_script = project_root / 'bot-dev/rl/ppo_train.py'
assert train_script.is_file(), f'Missing {train_script}; rerun the clone/setup cell and verify PROJECT_REF={PROJECT_REF!r}'
checked_out = subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'], cwd=project_root, check=True, text=True, capture_output=True).stdout.strip()
assert checked_out == PROJECT_REF, f'Expected branch {PROJECT_REF!r}, found {checked_out!r}; rerun the clone/setup cell'
TOTAL_STEPS = 2_000_000
args = [sys.executable,str(train_script),'--device','auto','--workers','4','--envs-per-worker','4',
        '--total-steps',str(TOTAL_STEPS),'--checkpoint-dir','/content/checkpoints',
        '--recovery-dir',RECOVERY,'--save-every-minutes','30'] + resume_args
print('Training from', train_script, 'on branch', checked_out)
subprocess.run(args, cwd=project_root, check=True)


In [ ]:
# Validation only. Do not put sealed-final seeds in this notebook.
pointer = json.load(open(os.path.join(RECOVERY, 'latest.json')))
checkpoint = os.path.join(RECOVERY, pointer['checkpoint'])
!mkdir -p /content/export src/code-here
!python bot-dev/rl/export_policy.py {checkpoint} /content/export/Robust_RL_v1.js
!python bot-dev/rl/export_policy_test.py {checkpoint} /content/export/Robust_RL_v1.js
!node bot-dev/rl/export_env_smoke.mjs /content/export/Robust_RL_v1.js
!node bot-dev/rl/eval/paired_eval.mjs --candidate=/content/export/Robust_RL_v1.js --output=/content/export/validation.jsonl
!python bot-dev/rl/eval/stats.py /content/export/validation.jsonl --output=/content/export/validation_stats.json
!node --expose-gc bot-dev/rl/eval/runtime_bench.mjs --candidate=/content/export/Robust_RL_v1.js > /content/export/runtime.json
print('Copy to src/code-here only after the pre-registered acceptance gate passes.')
